## 긱도, 각속도, 각가속도 계산

### 선택한 관절

0 : 코(머리 중심)
11, 12 : 왼쪽/오른쪽 어깨
13, 14 : 왼쪽/오른쪽 팔꿈치
15, 16 : 왼쪽/오른쪽 손목
23, 24 : 왼쪽/오른쪽 엉덩이(고관절-골반)
25, 26 : 왼쪽/오른쪽 무릎
27, 28 : 왼쪽/오른쪽 발목

In [83]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
# 라이브러리 할당
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [85]:

# ===========================================
# 1. 관절 각도 계산
# ===========================================
def calculate_angle(a, b, c, df, eps=1e-6):
    """
    세 점 (A, B, C)의 3D 좌표로 중심 B의 관절 각도를 계산 (degree)
    """
    a_xyz = df[[f"kp{a}_x", f"kp{a}_y", f"kp{a}_z"]].values
    b_xyz = df[[f"kp{b}_x", f"kp{b}_y", f"kp{b}_z"]].values
    c_xyz = df[[f"kp{c}_x", f"kp{c}_y", f"kp{c}_z"]].values

    # 벡터 계산
    ba = a_xyz - b_xyz
    bc = c_xyz - b_xyz

    # 내적/외적 계산
    dot = np.einsum('ij,ij->i', ba, bc)
    cross = np.linalg.norm(np.cross(ba, bc), axis=1)

    # 0 길이 벡터 방지 / eps:아주 작은 수(벡터가 0 혹은 거의 0)
    dot = np.where(np.abs(dot) < eps, eps, dot)
    cross = np.where(np.abs(cross) < eps, eps, cross)

    angle = np.degrees(np.arctan2(cross, dot))
    return angle




In [86]:
# ===========================================
# 2. 각속도 계산
# ===========================================
# 중앙 차분법 기반 각속도 계산 함수
def calculate_angular_velocity(angle_series, dt, fill_na=True, eps_dt=1e-6):
    """
    Mediapipe BlazePose 각도 시퀀스에서 각속도(°/s) 계산.
    중앙 차분법(central difference):
        ω_t = (θ_{t+1} - θ_{t-1}) / (2Δt)
    """
    # radian 단위로 변환 (계산 정확도 향상)
    # numpy array 또는 pandas Series 처리
    if isinstance(angle_series, pd.Series):
        theta = np.radians(angle_series.values)
    else:
        theta = np.radians(angle_series)  # ndarray 처리

    omega = np.zeros_like(theta)

    dt_arr = np.asarray(dt)
    if dt_arr.size == 1:
        dt_arr = dt_arr * np.ones_like(theta)

    # df=0 방지
    df_arr_safe = np.where(dt_arr == 0, eps_dt, dt_arr)

    # 중앙차분 적용 (끝 점은 0 유지)
    if len(theta) > 2:
        omega[1:-1] = (theta[2:] - theta[:-2]) / (2 * df_arr_safe[1:-1])

    # 다시 degree/sec 단위로 변환
    omega_deg = np.degrees(omega)

    # nan 값 채우기
    if fill_na:
      omega_deg = np.nan_to_num(omega_deg, nan=0.0, posinf=0.0, neginf=0.0)

    return omega_deg




In [87]:
# ===========================================
# 3. 각가속도 계산
# ===========================================
def calculate_angular_acceleration_safe(omega_series, dt, fill_na=True, eps_dt=1e-6):
  '''
  중앙 차분법 기반 각가속도 계산
  α_i = (ω_{i+1} - ω_{i-1}) / (2Δt)
  '''
  # pandas.Series이면 values로 변환, 아니면 그대로 사용
  if isinstance(omega_series, pd.Series):
      omega = omega_series.values.astype(float)
  else:
      omega = np.asarray(omega_series, dtype=float)

  alpha = np.zeros_like(omega, dtype=float)

  dt_arr = np.asarray(dt, dtype=float)
  if dt_arr.size == 1:
      dt_arr = dt_arr * np.ones_like(omega)
  dt_arr_safe = np.where(dt_arr == 0, eps_dt, dt_arr)

  if len(omega) > 2:
      alpha[1:-1] = (omega[2:] - omega[:-2]) / (2.0 * dt_arr_safe[1:-1])

  if fill_na:
      alpha = np.nan_to_num(alpha, nan=0.0, posinf=0.0, neginf=0.0)

  return alpha




In [88]:
# -------------------------------
# 4. dt 계산
# -------------------------------
def compute_dt(timestamps):
    timestamps = np.asarray(timestamps, dtype=float)
    dt_array = np.zeros_like(timestamps, dtype=float)
    dt_array[1:-1] = (timestamps[2:] - timestamps[:-2]) / 2.0
    dt_array[0] = timestamps[1] - timestamps[0]
    dt_array[-1] = timestamps[-1] - timestamps[-2]
    dt_array = np.where(dt_array == 0, 1e-6, dt_array)
    return dt_array

In [89]:
# 전처리가 다 완료된 파일 불러오기
input_path = "/content/drive/MyDrive/OnSafe/scaled_df.csv"
df = pd.read_csv(input_path, encoding='utf-8')

# 중복된 행 삭제
df = df.drop_duplicates()

In [90]:
df

,video,file_id,frame,timestamp,kp0_visibility,kp0_x,kp0_y,kp0_z,kp1_visibility,kp1_x,...,kp30_y,kp30_z,kp31_visibility,kp31_x,kp31_y,kp31_z,kp32_visibility,kp32_x,kp32_y,kp32_z
0,ADL,ADL_258_clip_06,0,0.000000,0.998187,0.267265,2.687150,-0.302766,0.998785,0.237088,...,-0.300039,1.634220,0.773355,-0.707502,-0.949614,0.570357,0.392821,-0.712404,-0.428392,2.002616
1,ADL,ADL_258_clip_06,1,0.016667,0.999278,0.320220,2.877692,-1.348117,0.999355,0.306190,...,-0.104569,1.741090,0.331871,-0.403819,-0.685467,1.520057,0.325424,-0.374196,-0.114031,2.381021
2,ADL,ADL_258_clip_06,2,0.033333,0.999675,0.364484,3.075438,-2.097327,0.999730,0.364482,...,0.120188,1.836470,0.556027,-0.096010,-0.357066,2.220579,0.328392,-0.089072,0.229840,2.613958
3,ADL,ADL_258_clip_06,3,0.050000,0.999524,0.401349,3.311254,-2.497695,0.999563,0.412561,...,0.377488,1.935926,0.308033,0.210114,0.039086,2.631184,0.376794,0.125769,0.605517,2.695004
4,ADL,ADL_258_clip_06,4,0.066667,0.999344,0.436084,3.721211,-2.336018,0.999298,0.446186,...,0.745653,2.063218,0.263840,0.483073,0.473342,2.351338,0.292943,0.152429,0.940984,2.341589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334611,FALL,S_D_0236_clip_00,25,0.833333,0.999983,-0.501979,-2.663398,-5.720356,0.999980,-0.425910,...,1.793530,-2.407830,0.659698,-0.008253,1.510209,0.867952,0.981766,-2.210674,1.613485,-3.555034
334612,FALL,S_D_0236_clip_00,26,0.866667,0.999983,-0.408866,-2.338803,-4.285218,0.999970,-0.347848,...,1.151231,-2.074305,0.557207,-0.309081,0.790296,0.942828,0.975555,-2.118559,0.767282,-2.952791
334613,FALL,S_D_0236_clip_00,27,0.900000,0.999380,-0.266984,-2.001678,-3.134834,0.999454,-0.216459,...,0.716810,-1.569578,0.558282,-0.632456,0.330036,0.194377,0.865539,-1.876665,0.295249,-2.244125
334614,FALL,S_D_0236_clip_00,28,0.933333,0.999671,-0.118551,-1.684378,-2.158500,0.999712,-0.075966,...,0.378526,-0.970146,0.798776,-0.932306,0.035434,-0.836689,0.891496,-1.589144,-0.002023,-1.467502


In [91]:
# -----------------------------------
# 관절 트리플
# -----------------------------------
joint_triplets = [
    # ('관절명', A, B, C)
    # -------------------------
    # 상체
    # -------------------------
    ('neck', 0, 11, 12), # 목
    ('shoulder_balance', 11, 0, 12), # 어깨 대칭성

    ('shoulder_left', 23, 11, 13),   # 왼쪽 엉덩이 - 어깨 - 팔꿈치
    ('shoulder_right', 24, 12, 14),  # 오른쪽 엉덩이 - 어깨 - 팔꿈치

    ('elbow_left', 11, 13, 15),      # 왼쪽 어깨 - 팔꿈치 - 손목
    ('elbow_right', 12, 14, 16),     # 오른쪽 어깨 - 팔꿈치 - 손목

    # -------------------------
    # 하체
    # -------------------------
    # 엉덩이
    ('hip_left', 11, 23, 25),        # 왼쪽 어깨 - 엉덩이 - 무릎
    ('hip_right', 12, 24, 26),       # 오른쪽 어깨 - 엉덩이 - 무릎

    ('knee_left', 23, 25, 27),       # 왼쪽 엉덩이 - 무릎 - 발목
    ('knee_right', 24, 26, 28),      # 오른쪽 엉덩이 - 무릎 - 발목

    # 발목
    ('ankle_left', 25, 27, 31),
    ('ankle_right', 26, 28, 32),

    # -------------------------
    # 중심축(몸통)
    # -------------------------
    # 상체 기울기
    ('torso_left', 0, 11, 23),  # 코 - 왼쪽 어깨 - 왼쪽 엉덩이
    ('torso_right', 0, 12, 24), # 코 - 오른쪽 어깨 - 오른쪽 엉덩이

    # 척추
    ('spine', 0, 23, 24),            # 코(머리 중심) - 왼엉덩이 - 오른엉덩이

]

In [92]:
# -----------------------------------
# 그룹별 처리 진행
# -----------------------------------
df_result = []



In [94]:
# -------------------------------
# 클립(video + file_id) 단위 계산
# -------------------------------
for (video, file_id), group in df.groupby(['video', 'file_id']):
    # 프레임이 너무 적으면 계산이 불가능하므로 건너뜁니다.
    if len(group) < 2:
        continue

    group = group.copy()
    dt_array = compute_dt(group['timestamp'].values)

    for name, a, b, c in joint_triplets:
        # 클립 단위 각도 계산
        angle = calculate_angle(a, b, c, group)
        # 각속도 계산
        omega = calculate_angular_velocity(angle, dt_array)
        # 각가속도 계산
        alpha = calculate_angular_acceleration_safe(omega, dt_array)

        # 컬럼에 저장
        group[f"{name}_angle"] = angle
        group[f"{name}_angular_velocity"] = omega
        group[f"{name}_angular_acceleration"] = alpha

    df_result.append(group)

In [95]:
df_final = pd.concat(df_result, axis=0).reset_index(drop=True)

In [96]:
# 신뢰도 관련 컬럼들은 삭제
df_final = df_final.loc[:, ~df_final.columns.str.contains('_visibility')]

In [97]:
df_final

,video,file_id,frame,timestamp,kp0_x,kp0_y,kp0_z,kp1_x,kp1_y,kp1_z,...,ankle_right_angular_acceleration,torso_left_angle,torso_left_angular_velocity,torso_left_angular_acceleration,torso_right_angle,torso_right_angular_velocity,torso_right_angular_acceleration,spine_angle,spine_angular_velocity,spine_angular_acceleration
0,ADL,ADL_100_clip_00,0,0.000000,-0.711991,-2.254846,0.336880,-0.765391,-2.417853,-0.025075,...,0.000000,71.808823,0.000000,0.000000,76.669725,0.000000,0.000000,68.111831,0.000000,0.000000
1,ADL,ADL_100_clip_00,1,0.016667,-0.711241,-0.609792,-0.135650,-0.741978,-0.682075,-0.308575,...,-8822.845456,60.950781,-148.169507,6298.371937,83.718524,409.671558,561.901210,64.598705,13.717789,-2055.415069
2,ADL,ADL_100_clip_00,2,0.033333,-0.747668,-0.275045,-0.243676,-0.771752,-0.325973,-0.369785,...,53051.507406,66.869839,209.945731,6401.296225,90.325444,18.730040,-18383.140230,68.569091,-68.513836,-6805.643321
3,ADL,ADL_100_clip_00,3,0.050000,-0.745503,-0.529014,-0.084298,-0.772894,-0.586615,-0.236518,...,12170.580699,67.948972,65.207034,-6382.819240,84.342859,-203.099783,144.630520,62.314911,-213.136989,3083.921846
4,ADL,ADL_100_clip_00,4,0.066667,-0.743692,-0.540675,-0.056276,-0.773123,-0.596327,-0.208209,...,-7018.747461,69.043407,-2.814910,-4085.725291,83.555451,23.551058,11449.122885,61.464525,34.283559,10712.091864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524692,FALL,Subject4_Fall_17_clip_06,22,0.758621,1.305631,0.302685,0.167546,1.387997,0.305160,0.327900,...,696.040891,52.481714,3.939694,211.089303,98.757268,18.204234,3201.761004,88.428057,-1.369551,11.409774
524693,FALL,Subject4_Fall_17_clip_06,23,0.793103,1.288327,0.273986,0.192223,1.368168,0.278079,0.345149,...,633.252007,53.009040,7.748418,388.405817,100.656365,187.421033,-917.948283,88.313145,21.046900,-529.589874
524694,FALL,Subject4_Fall_17_clip_06,24,0.827586,1.150079,0.429645,0.229989,1.215883,0.452959,0.375112,...,-79.421324,53.016088,30.726302,7214.034631,111.682856,-45.102544,-16573.473218,89.879567,-37.892991,-9117.846468
524695,FALL,Subject4_Fall_17_clip_06,25,0.862069,1.373448,0.100675,0.177170,1.456233,0.102185,0.327517,...,-161.675383,55.128096,505.268048,-445.531380,97.545845,-955.577120,653.986887,85.699835,-607.770097,549.448370


In [98]:
# csv 파일 할당
df = df_final

In [99]:
# 중심 좌표
def compute_center_coordinates(df, left_pelvis='kp23', right_pelvis='kp24'):
    """
    좌/우 골반 평균으로 중심 좌표 계산
    """
    df = df.copy()
    df['center_x'] = (df[f'{left_pelvis}_x'] + df[f'{right_pelvis}_x']) / 2
    df['center_y'] = (df[f'{left_pelvis}_y'] + df[f'{right_pelvis}_y']) / 2
    df['center_z'] = (df[f'{left_pelvis}_z'] + df[f'{right_pelvis}_z']) / 2
    return df

In [100]:
# 몸 중심 이동 거리 계산 : 골반
def center_displacement(df, kp1='kp23', kp2='kp24'):
    coords = (df[[f"{kp1}_x", f"{kp1}_y", f"{kp1}_z"]].values +
              df[[f"{kp2}_x", f"{kp2}_y", f"{kp2}_z"]].values) / 2
    if len(coords) < 2: return 0
    diff = np.diff(coords, axis=0)
    dist = np.linalg.norm(diff, axis=1)
    return np.sum(dist)

In [101]:
# 몸 중심 속도 변화량 계산
def center_velocity_change(df, kp1='kp23', kp2='kp24', dt_array=None, eps=1e-6):
    coords = (df[[f'{kp1}_x', f'{kp1}_y', f'{kp1}_z']].values +
              df[[f'{kp2}_x', f'{kp2}_y', f'{kp2}_z']].values) / 2
    if len(coords) < 2: return 0

    diff = np.diff(coords, axis=0)
    # dt_array가 전체 길이라면 diff 길이에 맞게 슬라이싱 (후행 차분 기준)
    if dt_array is None:
        dt = np.ones(len(diff))
    else:
        dt = dt_array[1:]

    dt = np.where(dt == 0, eps, dt)
    vel = np.linalg.norm(diff, axis=1) / dt

    if len(vel) < 2: return 0
    return np.sum(np.abs(np.diff(vel)))

In [102]:
# 중심 이동 속도 및 가속도 계산 (데이터프레임 반환용)
def compute_center_metrics(df, kp1='kp23', kp2='kp24'):
    df = df.copy()
    coords = (df[[f'{kp1}_x', f'{kp1}_y', f'{kp1}_z']].values +
              df[[f'{kp2}_x', f'{kp2}_y', f'{kp2}_z']].values) / 2

    # 거리 계산
    diff = np.diff(coords, axis=0, prepend=coords[:1])
    df['center_distance'] = np.linalg.norm(diff, axis=1)

    # 시간 차이
    timestamps = df['timestamp'].values
    dt = np.diff(timestamps, prepend=timestamps[0])
    dt = np.where(dt == 0, 1e-6, dt)

    # 속도 및 가속도
    df['center_speed'] = df['center_distance'] / dt
    speed_diff = np.diff(df['center_speed'], prepend=df['center_speed'].iloc[0])
    df['center_acceleration'] = speed_diff / dt

    return df

In [103]:
# ===========================================
# 실행부 (수정된 루프)
# ===========================================
df_result = []

In [104]:
for (video, file_id), group in df.groupby(['video', 'file_id']):
    group = group.sort_values('frame').copy()

    # 1. 프레임별 속도/가속도/거리 계산 (행 단위 결과)
    group = compute_center_metrics(group)
    group = compute_center_coordinates(group)


    df_result.append(group)

# 모든 frame 합치기
df_final = pd.concat(df_result, axis=0).reset_index(drop=True)

In [105]:
# ===========================================
# 모든 frame 합치기
# ===========================================
print(f"원본 df shape: {df.shape}")
print(f"프레임 단위 피처 df_frame shape: {df_final.shape}")

원본 df shape: (524697, 148)
프레임 단위 피처 df_frame shape: (524697, 154)


In [106]:
df

,video,file_id,frame,timestamp,kp0_x,kp0_y,kp0_z,kp1_x,kp1_y,kp1_z,...,ankle_right_angular_acceleration,torso_left_angle,torso_left_angular_velocity,torso_left_angular_acceleration,torso_right_angle,torso_right_angular_velocity,torso_right_angular_acceleration,spine_angle,spine_angular_velocity,spine_angular_acceleration
0,ADL,ADL_100_clip_00,0,0.000000,-0.711991,-2.254846,0.336880,-0.765391,-2.417853,-0.025075,...,0.000000,71.808823,0.000000,0.000000,76.669725,0.000000,0.000000,68.111831,0.000000,0.000000
1,ADL,ADL_100_clip_00,1,0.016667,-0.711241,-0.609792,-0.135650,-0.741978,-0.682075,-0.308575,...,-8822.845456,60.950781,-148.169507,6298.371937,83.718524,409.671558,561.901210,64.598705,13.717789,-2055.415069
2,ADL,ADL_100_clip_00,2,0.033333,-0.747668,-0.275045,-0.243676,-0.771752,-0.325973,-0.369785,...,53051.507406,66.869839,209.945731,6401.296225,90.325444,18.730040,-18383.140230,68.569091,-68.513836,-6805.643321
3,ADL,ADL_100_clip_00,3,0.050000,-0.745503,-0.529014,-0.084298,-0.772894,-0.586615,-0.236518,...,12170.580699,67.948972,65.207034,-6382.819240,84.342859,-203.099783,144.630520,62.314911,-213.136989,3083.921846
4,ADL,ADL_100_clip_00,4,0.066667,-0.743692,-0.540675,-0.056276,-0.773123,-0.596327,-0.208209,...,-7018.747461,69.043407,-2.814910,-4085.725291,83.555451,23.551058,11449.122885,61.464525,34.283559,10712.091864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524692,FALL,Subject4_Fall_17_clip_06,22,0.758621,1.305631,0.302685,0.167546,1.387997,0.305160,0.327900,...,696.040891,52.481714,3.939694,211.089303,98.757268,18.204234,3201.761004,88.428057,-1.369551,11.409774
524693,FALL,Subject4_Fall_17_clip_06,23,0.793103,1.288327,0.273986,0.192223,1.368168,0.278079,0.345149,...,633.252007,53.009040,7.748418,388.405817,100.656365,187.421033,-917.948283,88.313145,21.046900,-529.589874
524694,FALL,Subject4_Fall_17_clip_06,24,0.827586,1.150079,0.429645,0.229989,1.215883,0.452959,0.375112,...,-79.421324,53.016088,30.726302,7214.034631,111.682856,-45.102544,-16573.473218,89.879567,-37.892991,-9117.846468
524695,FALL,Subject4_Fall_17_clip_06,25,0.862069,1.373448,0.100675,0.177170,1.456233,0.102185,0.327517,...,-161.675383,55.128096,505.268048,-445.531380,97.545845,-955.577120,653.986887,85.699835,-607.770097,549.448370


In [107]:
df_final

,video,file_id,frame,timestamp,kp0_x,kp0_y,kp0_z,kp1_x,kp1_y,kp1_z,...,torso_right_angular_acceleration,spine_angle,spine_angular_velocity,spine_angular_acceleration,center_distance,center_speed,center_acceleration,center_x,center_y,center_z
0,ADL,ADL_100_clip_00,0,0.000000,-0.711991,-2.254846,0.336880,-0.765391,-2.417853,-0.025075,...,0.000000,68.111831,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
1,ADL,ADL_100_clip_00,0,0.000000,-0.711991,-2.254846,0.336880,-0.765391,-2.417853,-0.025075,...,0.000000,68.111831,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
2,ADL,ADL_100_clip_00,1,0.016667,-0.711241,-0.609792,-0.135650,-0.741978,-0.682075,-0.308575,...,561.901210,64.598705,13.717789,-2055.415069,1.942890e-16,1.165734e-14,6.994405e-13,0.000000e+00,-1.942890e-16,0.0
3,ADL,ADL_100_clip_00,1,0.016667,-0.711241,-0.609792,-0.135650,-0.741978,-0.682075,-0.308575,...,561.901210,64.598705,13.717789,-2055.415069,0.000000e+00,0.000000e+00,-1.165734e-08,0.000000e+00,-1.942890e-16,0.0
4,ADL,ADL_100_clip_00,2,0.033333,-0.747668,-0.275045,-0.243676,-0.771752,-0.325973,-0.369785,...,-18383.140230,68.569091,-68.513836,-6805.643321,3.955170e-16,2.373102e-14,1.423861e-12,0.000000e+00,2.012279e-16,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524692,FALL,Subject4_Fall_17_clip_06,22,0.758621,1.305631,0.302685,0.167546,1.387997,0.305160,0.327900,...,3201.761004,88.428057,-1.369551,11.409774,2.498002e-16,7.244205e-15,-2.100820e-13,0.000000e+00,-4.996004e-16,0.0
524693,FALL,Subject4_Fall_17_clip_06,23,0.793103,1.288327,0.273986,0.192223,1.368168,0.278079,0.345149,...,-917.948283,88.313145,21.046900,-529.589874,3.532708e-16,1.024485e-14,8.701879e-14,2.498002e-16,-2.498002e-16,0.0
524694,FALL,Subject4_Fall_17_clip_06,24,0.827586,1.150079,0.429645,0.229989,1.215883,0.452959,0.375112,...,-16573.473218,89.879567,-37.892991,-9117.846468,3.532708e-16,1.024485e-14,9.608326e-28,0.000000e+00,0.000000e+00,0.0
524695,FALL,Subject4_Fall_17_clip_06,25,0.862069,1.373448,0.100675,0.177170,1.456233,0.102185,0.327517,...,653.986887,85.699835,-607.770097,549.448370,4.170847e-16,1.209546e-14,5.366746e-14,2.983724e-16,2.914335e-16,0.0


In [108]:
# 좌표 컬럼 삭제
coord_cols = [c for c in df_final.columns if ('_x' in c or '_y' in c or '_z' in c)]
df_final = df_final.drop(columns=coord_cols)
df_final.shape

(524697, 52)

In [109]:
for col in df_final.columns:
    print(col)

video
file_id
frame
timestamp
neck_angle
neck_angular_velocity
neck_angular_acceleration
shoulder_balance_angle
shoulder_balance_angular_velocity
shoulder_balance_angular_acceleration
shoulder_left_angle
shoulder_left_angular_velocity
shoulder_left_angular_acceleration
shoulder_right_angle
shoulder_right_angular_velocity
shoulder_right_angular_acceleration
elbow_left_angle
elbow_left_angular_velocity
elbow_left_angular_acceleration
elbow_right_angle
elbow_right_angular_velocity
elbow_right_angular_acceleration
hip_left_angle
hip_left_angular_velocity
hip_left_angular_acceleration
hip_right_angle
hip_right_angular_velocity
hip_right_angular_acceleration
knee_left_angle
knee_left_angular_velocity
knee_left_angular_acceleration
knee_right_angle
knee_right_angular_velocity
knee_right_angular_acceleration
ankle_left_angle
ankle_left_angular_velocity
ankle_left_angular_acceleration
ankle_right_angle
ankle_right_angular_velocity
ankle_right_angular_acceleration
torso_left_angle
torso_left_ang

In [110]:
# 컬럼 순서 변경
joints = ['neck', 'shoulder_balance', 'shoulder_left', 'shoulder_right', 'elbow_left', 'elbow_right',
                  'hip_left', 'hip_right', 'knee_left', 'knee_right', 'torso_left', 'torso_right', 'spine']

features_order = [
    'angle',
    'angular_velocity',
    'angular_acceleration',
    'fast_ratio',
    'stationary_ratio',
    'peak_interval'
]
# 컬럼 순서 리스트 만들기
new_columns = ['video', 'file_id', 'frame', 'timestamp']
# print(new_cols)


for joint in joints:
    for feat in features_order:
        col_name = f"{joint}_{feat}"
        if col_name in df_final.columns:
            new_columns.append(col_name)

# 남는 컬럼은 마지막에 추가
for col in df_final.columns.tolist():
    if col not in new_columns:
        new_columns.append(col)

df_final = df_final[new_columns]


In [111]:
for col in df_final.columns:
    print(col)

video
file_id
frame
timestamp
neck_angle
neck_angular_velocity
neck_angular_acceleration
shoulder_balance_angle
shoulder_balance_angular_velocity
shoulder_balance_angular_acceleration
shoulder_left_angle
shoulder_left_angular_velocity
shoulder_left_angular_acceleration
shoulder_right_angle
shoulder_right_angular_velocity
shoulder_right_angular_acceleration
elbow_left_angle
elbow_left_angular_velocity
elbow_left_angular_acceleration
elbow_right_angle
elbow_right_angular_velocity
elbow_right_angular_acceleration
hip_left_angle
hip_left_angular_velocity
hip_left_angular_acceleration
hip_right_angle
hip_right_angular_velocity
hip_right_angular_acceleration
knee_left_angle
knee_left_angular_velocity
knee_left_angular_acceleration
knee_right_angle
knee_right_angular_velocity
knee_right_angular_acceleration
torso_left_angle
torso_left_angular_velocity
torso_left_angular_acceleration
torso_right_angle
torso_right_angular_velocity
torso_right_angular_acceleration
spine_angle
spine_angular_veloc

In [112]:
df_final.shape

(524697, 52)

In [113]:
# csv 파일 만들기
df_final.to_csv("/content/drive/MyDrive/OnSafe/make_feature.csv", index=False, encoding='utf-8')